# PoolCue Vision Assist — YOLOv11 Training
**AIPI 590 Final Project**

Run this notebook in Google Colab (free T4 GPU).  
Runtime → Change runtime type → T4 GPU

### Steps
1. Install dependencies
2. Pull merged dataset from Roboflow
3. Train YOLOv11n (nano — fastest, runs on Pi)
4. Evaluate
5. Export to ONNX for Pi deployment
6. Download `best.pt` and `best.onnx`

In [ ]:
# Step 1: Install
!pip install ultralytics roboflow -q

In [ ]:
# Step 2: Download dataset from Roboflow
# Replace API_KEY with your Roboflow API key
# Replace workspace/project/version with your merged dataset details

from roboflow import Roboflow

API_KEY = "YOUR_ROBOFLOW_API_KEY"   # <-- paste your key here
WORKSPACE = "billiard-ball-data-set" # <-- your workspace slug
PROJECT   = "billiard-pool"          # <-- your project slug
VERSION   = 1                        # <-- dataset version number

rf = Roboflow(api_key=API_KEY)
project = rf.workspace(WORKSPACE).project(PROJECT)
dataset = project.version(VERSION).download("yolov11")

print("Dataset location:", dataset.location)
print("Classes:", project.version(VERSION).classes)

In [ ]:
# Step 3: Check dataset structure
import os, yaml

data_yaml = os.path.join(dataset.location, "data.yaml")
with open(data_yaml) as f:
    data_cfg = yaml.safe_load(f)

print(yaml.dump(data_cfg, default_flow_style=False))
print("Train images:", len(os.listdir(os.path.join(dataset.location, "train", "images"))))
print("Val images:  ", len(os.listdir(os.path.join(dataset.location, "valid", "images"))))

In [ ]:
# Step 4: Train YOLOv11 nano
# yolo11n = nano model — fastest inference, small enough for Pi
from ultralytics import YOLO

model = YOLO("yolo11n.pt")  # downloads pretrained nano weights

results = model.train(
    data=data_yaml,
    epochs=80,
    imgsz=640,
    batch=16,
    patience=20,          # early stopping
    project="pool_vision",
    name="yolo11n_pool",
    device=0,             # GPU
    augment=True,
    hsv_h=0.015,          # slight hue shift — helps with different table felt colors
    hsv_s=0.4,
    hsv_v=0.4,
    flipud=0.0,            # don't flip vertically (overhead camera, orientation matters)
    fliplr=0.5,
    mosaic=1.0,
)

In [ ]:
# Step 5: Evaluate on validation set
best_model = YOLO("pool_vision/yolo11n_pool/weights/best.pt")
metrics = best_model.val(data=data_yaml)

print(f"mAP50:    {metrics.box.map50:.3f}")
print(f"mAP50-95: {metrics.box.map:.3f}")
print(f"Precision:{metrics.box.mp:.3f}")
print(f"Recall:   {metrics.box.mr:.3f}")

In [ ]:
# Step 6: Visualize predictions on a few val images
import glob, random
from IPython.display import Image, display

val_images = glob.glob(os.path.join(dataset.location, "valid", "images", "*.jpg"))
sample = random.sample(val_images, min(3, len(val_images)))

for img_path in sample:
    result = best_model(img_path, conf=0.45)[0]
    out_path = img_path.replace(".jpg", "_pred.jpg")
    result.save(filename=out_path)
    display(Image(out_path, width=600))

In [ ]:
# Step 7: Export to ONNX (faster CPU inference on Pi)
best_model.export(format="onnx", imgsz=640, simplify=True)
print("ONNX model saved alongside best.pt")

In [ ]:
# Step 8: Download weights to your machine
from google.colab import files

files.download("pool_vision/yolo11n_pool/weights/best.pt")
files.download("pool_vision/yolo11n_pool/weights/best.onnx")
print("Downloaded best.pt and best.onnx")
print("Place best.pt in: models/pool_vision/best.pt")

In [ ]:
# Step 9: Print class names (copy these into src/vision/detector.py BALL_CLASSES)
print("Class names from your dataset:")
for i, name in enumerate(best_model.names.values()):
    print(f"  {i}: {name}")

print("\nCopy the list above into BALL_CLASSES in src/vision/detector.py")